# Semantic and Hybrid Search: BM25, Dense Retrieval and Rank Fusion

The runnable code of [Semantic and Hybrid Search: BM25, Dense Retrieval and Rank Fusion](https://residual-stream.lvtan71.workers.dev/posts/semantic-and-hybrid-search/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/semantic-and-hybrid-search.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 2 min end to end.
The post's run read a copy of the blog taken mid-session, with a draft of post 36 that isn't in git; this rebuild has 308 English sections where the post had 309, so the last digits can differ.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "semantic-and-hybrid-search"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/semantic-and-hybrid-search/retrieval.py', 'shared/semantic-and-hybrid-search/sections.py', 'data/blog-post37.tar.gz']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## BM25: keyword search, derived

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/semantic-and-hybrid-search/#bm25-keyword-search-derived)

In [ ]:
import math
from collections import Counter
exec(open("retrieval.py").read().split("models = {")[0])       # sections, tokens(), BM25

# BM25 worked for one query on this blog's 309 English sections: the
# query "paged KV cache", and the section that scores highest for it.
docs = [s["body"] for s in en]
bm = BM25(docs)
q = "paged KV cache"
best = int(bm.scores(q).argmax())
d = bm.docs[best]
print(f'query "{q}"; best section: "{en[best]["heading"]}" ({en[best]["post"]}), {bm.len[best]} words, '
      f"average {bm.avg:.0f}")
total = 0
for t in tokens(q):
    n = sum(1 for doc in bm.docs if t in doc)
    tf = d[t]
    norm = bm.k1 * (1 - bm.b + bm.b * bm.len[best] / bm.avg)
    part = bm.idf[t] * tf * (bm.k1 + 1) / (tf + norm)
    total += part
    print(f"  {t:6s} in {n:3d} of {len(docs)} sections: idf {bm.idf[t]:.2f}; appears {tf:2d} times here; "
          f"saturated tf {tf * (bm.k1 + 1) / (tf + norm):.2f}; contributes {part:.2f}")
print(f"  score {total:.2f}")

The post printed:

```text
query "paged KV cache"; best section: "Borrowing an idea from operating systems" (paged-attention-explained), 231 words, average 280
  paged  in  16 of 309 sections: idf 2.93; appears  2 times here; saturated tf 1.51; contributes 4.44
  kv     in  46 of 309 sections: idf 1.90; appears  4 times here; saturated tf 1.89; contributes 3.58
  cache  in  73 of 309 sections: idf 1.44; appears  2 times here; saturated tf 1.51; contributes 2.18
  score 10.20
```

## A benchmark from this blog

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/semantic-and-hybrid-search/#a-benchmark-from-this-blog)

In [ ]:
import collections
import math
import re
import unicodedata
import numpy as np
from sentence_transformers import SentenceTransformer
from sections import sections

# Keyword (BM25), dense and hybrid retrieval on this blog's own sections:
# each heading is a query whose one relevant document is its section's text.
# Headings that occur more than once in a language are left out (ambiguous).
def unique(secs):
    count = collections.Counter(s["heading"].lower() for s in secs)
    return [s for s in secs if count[s["heading"].lower()] == 1]
en, vi = sections("en"), sections("vi")
def strip_marks(t):                                  # Vietnamese typed without diacritics
    t = t.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")


# Cross-lingual pairs: a Vietnamese heading and the English section of the
# same post whose heading it translates, matched as mutual best matches by
# the multilingual model's heading similarity (the twins' sections don't
# always line up one to one).
def cross_pairs():
    by_post = lambda secs: {p: [s for s in secs if s["post"] == p and s in unique(secs)] for p in {s["post"] for s in secs}}
    EN, VI = by_post(en), by_post(vi)
    pairs = []
    for p in VI:
        if not EN.get(p) or not VI[p]:
            continue
        S = models["MiniLM (multilingual)"].encode([v["heading"] for v in VI[p]], normalize_embeddings=True) @ \
            models["MiniLM (multilingual)"].encode([e["heading"] for e in EN[p]], normalize_embeddings=True).T
        for i in range(len(VI[p])):
            j = int(S[i].argmax())
            if int(S[:, j].argmax()) == i:
                pairs.append((VI[p][i]["heading"], en.index(EN[p][j])))
    return pairs

def tokens(t):
    return re.findall(r"\w+", t.lower())

class BM25:
    """Okapi BM25 (Robertson and Zaragoza, 2009), k1 = 1.5, b = 0.75."""
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [collections.Counter(tokens(d)) for d in docs]
        self.len = np.array([sum(d.values()) for d in self.docs]); self.avg = self.len.mean()
        df = collections.Counter(t for d in self.docs for t in d)
        n = len(docs)
        self.idf = {t: math.log((n - f + 0.5) / (f + 0.5) + 1) for t, f in df.items()}
        self.k1, self.b = k1, b
    def scores(self, query):
        s = np.zeros(len(self.docs))
        for t in set(tokens(query)):
            if t not in self.idf:
                continue
            tf = np.array([d[t] for d in self.docs])
            s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.len / self.avg))
        return s

models = {"MiniLM (English)": SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu"),
          "MiniLM (multilingual)": SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")}
def dense(model, docs):
    """One vector per section. The models read at most 256 (English) and
    128 (multilingual) word pieces, so long sections are cut off."""
    D = models[model].encode(docs, batch_size=64, normalize_embeddings=True)
    return lambda q: D @ models[model].encode([q], normalize_embeddings=True)[0]

def dense_passages(model, docs, words=80):
    """One vector per 80-word passage; a section scores its best passage."""
    owner, passages = [], []
    for i, d in enumerate(docs):
        w = d.split()
        for j in range(0, len(w), words):
            owner.append(i); passages.append(" ".join(w[j:j + words]))
    P = models[model].encode(passages, batch_size=64, normalize_embeddings=True)
    owner = np.array(owner)
    def f(q):
        s = P @ models[model].encode([q], normalize_embeddings=True)[0]
        best = np.full(len(docs), -np.inf)
        np.maximum.at(best, owner, s)
        return best
    return f

def rrf(*score_fns, k=60, weights=None):
    """Reciprocal rank fusion (Cormack et al., 2009): sum of 1 / (k + rank),
    optionally weighted per ranker."""
    weights = weights or [1] * len(score_fns)
    def f(q):
        total = 0
        for fn, wt in zip(score_fns, weights):
            order = np.argsort(-fn(q)); rank = np.empty_like(order); rank[order] = np.arange(1, len(order) + 1)
            total = total + wt / (k + rank)
        return total
    return f

def evaluate(name, fn, queries):
    ranks = []
    for q, target in queries:
        s = fn(q)
        ranks.append(int((s >= s[target]).sum()))                     # its rank; ties counted against it
    r = np.array(ranks)
    print(f"  {name:38s} recall@1 {np.mean(r == 1):5.0%}  recall@10 {np.mean(r <= 10):5.0%}  "
          f"MRR {np.mean(1 / r):.3f}  nDCG@10 {np.mean(np.where(r <= 10, 1 / np.log2(r + 1), 0)):.3f}")
    return 1 / r

def paired(name, a, b):
    """MRR difference a - b, with a 95% bootstrap interval over queries."""
    rng = np.random.default_rng(0)
    d = a - b
    boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"  {name}: MRR difference {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]")

# How much each encoder reads: the share of sections and 80-word passages
# longer than its input limit (256 word pieces English, 128 multilingual).
for lang, secs in (("English", en), ("Vietnamese", vi)):
    for name, m in models.items():
        tk, lim = m.tokenizer, m.max_seq_length
        docs = [s["body"] for s in secs]
        passages = [" ".join(d.split()[j:j + 80]) for d in docs for j in range(0, len(d.split()), 80)]
        cut = lambda xs: np.mean([len(tk(x).input_ids) > lim for x in xs])
        print(f"{lang} text, {name}: {cut(docs):.0%} of sections and {cut(passages):.0%} of passages cut off")
cross = cross_pairs()

for title, corpus, queries in (
        ("English headings -> English sections", en, [(s["heading"], i) for i, s in enumerate(en) if s in unique(en)]),
        ("Vietnamese headings -> Vietnamese sections", vi, [(s["heading"], i) for i, s in enumerate(vi) if s in unique(vi)]),
        ("Vietnamese headings typed without diacritics -> Vietnamese sections", vi,
         [(strip_marks(s["heading"]), i) for i, s in enumerate(vi) if s in unique(vi)]),
        ("Vietnamese headings -> English sections (cross-lingual, headings matched)", en, cross)):
    docs = [s["body"] for s in corpus]
    print(f"\n{title}: {len(queries)} queries, {len(docs)} sections")
    bm25 = BM25(docs).scores
    d_en, d_multi = dense("MiniLM (English)", docs), dense("MiniLM (multilingual)", docs)
    p_en, p_multi = dense_passages("MiniLM (English)", docs), dense_passages("MiniLM (multilingual)", docs)
    rr_bm25 = evaluate("BM25", bm25, queries)
    evaluate("dense, English, whole section", d_en, queries)
    evaluate("dense, English, best passage", p_en, queries)
    evaluate("dense, multilingual, whole section", d_multi, queries)
    evaluate("dense, multilingual, best passage", p_multi, queries)
    rr_en = evaluate("hybrid: BM25 + English passages", rrf(bm25, p_en), queries)
    rr_multi = evaluate("hybrid: BM25 + multilingual passages", rrf(bm25, p_multi), queries)
    paired("hybrid (English) minus BM25", rr_en, rr_bm25)
    paired("hybrid (multilingual) minus BM25", rr_multi, rr_bm25)
    evaluate("hybrid, multilingual weighted 0.25", rrf(bm25, p_multi, weights=[1, 0.25]), queries)

# The fix for missing diacritics: index a stripped copy too, and search it
# with stripped queries.
vi_q = [(strip_marks(s["heading"]), i) for i, s in enumerate(vi) if s in unique(vi)]
print("\nVietnamese headings typed without diacritics, against a diacritic-stripped index:")
evaluate("BM25, stripped index", BM25([strip_marks(s["body"]) for s in vi]).scores, vi_q)

The post printed:

```text
English text, MiniLM (English): 72% of sections and 0% of passages cut off
English text, MiniLM (multilingual): 94% of sections and 48% of passages cut off
Vietnamese text, MiniLM (English): 89% of sections and 0% of passages cut off
Vietnamese text, MiniLM (multilingual): 96% of sections and 14% of passages cut off

English headings -> English sections: 297 queries, 309 sections
  BM25                                   recall@1   35%  recall@10   73%  MRR 0.480  nDCG@10 0.535
  dense, English, whole section          recall@1   31%  recall@10   72%  MRR 0.440  nDCG@10 0.499
  dense, English, best passage           recall@1   32%  recall@10   71%  MRR 0.458  nDCG@10 0.511
  dense, multilingual, whole section     recall@1   27%  recall@10   60%  MRR 0.374  nDCG@10 0.419
  dense, multilingual, best passage      recall@1   26%  recall@10   64%  MRR 0.380  nDCG@10 0.432
  hybrid: BM25 + English passages        recall@1   40%  recall@10   77%  MRR 0.530  nDCG@10 0.582
  hybrid: BM25 + multilingual passages   recall@1   35%  recall@10   75%  MRR 0.494  nDCG@10 0.550
  hybrid (English) minus BM25: MRR difference +0.051 [+0.016, +0.086]
  hybrid (multilingual) minus BM25: MRR difference +0.015 [-0.024, +0.052]
  hybrid, multilingual weighted 0.25     recall@1   38%  recall@10   74%  MRR 0.516  nDCG@10 0.566

Vietnamese headings -> Vietnamese sections: 257 queries, 270 sections
  BM25                                   recall@1   42%  recall@10   78%  MRR 0.539  nDCG@10 0.591
  dense, English, whole section          recall@1   17%  recall@10   44%  MRR 0.262  nDCG@10 0.294
  dense, English, best passage           recall@1   16%  recall@10   48%  MRR 0.256  nDCG@10 0.297
  dense, multilingual, whole section     recall@1   25%  recall@10   54%  MRR 0.352  nDCG@10 0.386
  dense, multilingual, best passage      recall@1   18%  recall@10   56%  MRR 0.308  nDCG@10 0.358
  hybrid: BM25 + English passages        recall@1   28%  recall@10   67%  MRR 0.414  nDCG@10 0.465
  hybrid: BM25 + multilingual passages   recall@1   35%  recall@10   75%  MRR 0.481  nDCG@10 0.540
  hybrid (English) minus BM25: MRR difference -0.125 [-0.168, -0.085]
  hybrid (multilingual) minus BM25: MRR difference -0.058 [-0.102, -0.015]
  hybrid, multilingual weighted 0.25     recall@1   40%  recall@10   77%  MRR 0.537  nDCG@10 0.588

Vietnamese headings typed without diacritics -> Vietnamese sections: 257 queries, 270 sections
  BM25                                   recall@1   26%  recall@10   58%  MRR 0.366  nDCG@10 0.411
  dense, English, whole section          recall@1   18%  recall@10   44%  MRR 0.260  nDCG@10 0.290
  dense, English, best passage           recall@1   16%  recall@10   49%  MRR 0.255  nDCG@10 0.299
  dense, multilingual, whole section     recall@1   14%  recall@10   34%  MRR 0.209  nDCG@10 0.230
  dense, multilingual, best passage      recall@1    9%  recall@10   30%  MRR 0.165  nDCG@10 0.184
  hybrid: BM25 + English passages        recall@1   20%  recall@10   59%  MRR 0.336  nDCG@10 0.388
  hybrid: BM25 + multilingual passages   recall@1   20%  recall@10   54%  MRR 0.314  nDCG@10 0.357
  hybrid (English) minus BM25: MRR difference -0.030 [-0.066, +0.007]
  hybrid (multilingual) minus BM25: MRR difference -0.053 [-0.089, -0.016]
  hybrid, multilingual weighted 0.25     recall@1   26%  recall@10   59%  MRR 0.368  nDCG@10 0.414

Vietnamese headings -> English sections (cross-lingual, headings matched): 245 queries, 309 sections
  BM25                                   recall@1   27%  recall@10   60%  MRR 0.376  nDCG@10 0.426
  dense, English, whole section          recall@1   13%  recall@10   40%  MRR 0.205  nDCG@10 0.238
  dense, English, best passage           recall@1   13%  recall@10   38%  MRR 0.217  nDCG@10 0.243
  dense, multilingual, whole section     recall@1   27%  recall@10   55%  MRR 0.363  nDCG@10 0.396
  dense, multilingual, best passage      recall@1   21%  recall@10   58%  MRR 0.331  nDCG@10 0.382
  hybrid: BM25 + English passages        recall@1   22%  recall@10   58%  MRR 0.339  nDCG@10 0.389
  hybrid: BM25 + multilingual passages   recall@1   26%  recall@10   62%  MRR 0.385  nDCG@10 0.434
  hybrid (English) minus BM25: MRR difference -0.037 [-0.077, +0.002]
  hybrid (multilingual) minus BM25: MRR difference +0.009 [-0.031, +0.049]
  hybrid, multilingual weighted 0.25     recall@1   31%  recall@10   62%  MRR 0.423  nDCG@10 0.465

Vietnamese headings typed without diacritics, against a diacritic-stripped index:
  BM25, stripped index                   recall@1   40%  recall@10   76%  MRR 0.524  nDCG@10 0.576
```